# 06. LightGBM 입력 — 구간별 parquet

한 행 = 거래 1건. **GNN 이 그 거래에서 보는 것 중 그래프 이웃을 뺀 전부**를 붙인다.

| 열 | 내용 |
|---|---|
| `edge_id, day, y, cls, w` | 라벨(`cls` 0 정상 / 1~8 패턴 / 9 무패턴), 표본 가중치 |
| 엣지 17 + `payment_format_code` + `in_port`, `out_port` | 04 출력 |
| `src_*` 19, `dst_*` 19 | 그 거래 날짜의 마감 스냅샷에서 송신·수신 계좌 행 (05 출력) |

`dv_lib.GNN_PARITY_COLUMNS` (55열) = GNN 입력과 같은 정보량 (엣지 14 + 결제유형 + port 2 + 노드 38).
`dv_lib.MODEL_COLUMNS` (58열) = 여기에 GNN 이 뺀 3개(`pair_history_count`, `same_entity`, `amt_z_pay_ccy`)를 더한 것.

| 구간 | 날짜 (양끝 포함) | 거래 수 | 저장 |
|---|---|---|---|
| train | 08-08 ~ 10-02 | 102,336,629 | 양성 전부 + 정상 `TRAIN_NEG_RATE` 표본, `w = 1/TRAIN_NEG_RATE` |
| val | 10-03 ~ 10-16 | 24,413,046 | binary_classification 과 같은 검증 표본 (양성 전부 + 정상 2%, 날짜별 가중치) — 조기 종료·임계값 |
| test | 10-17 ~ 10-30 | 26,808,230 | 전부 |

train 정상 표본은 날짜별 고정 난수(`seed`, 날짜)라 다시 돌려도 같다.
val 은 `parquets/<DATASET>/sampling/val/2022-10-03_2022-10-16_neg0.02.parquet`(binary_classification 이 만든 파일)을 그대로 쓰고,
없으면 같은 규칙으로 만든다 → 검증 지표를 binary_classification 결과와 바로 나란히 놓을 수 있다.

In [ ]:
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path.cwd()))                    # code/nonpaten_vs_normal 에서 연다
import importlib, dv_lib as D; importlib.reload(D)

DATASET = "HI-Large"
THREADS, MEMORY = 8, "10GB"        # DuckDB. 컨테이너 한도(약 25GB) 안에서 파이썬 배열과 나눠 쓰도록 절반 이하
P = D.Paths(DATASET)
print(P)

TRAIN_NEG_RATE = 0.1              # train: 양성 전부 + 정상 10% (w = 10)
VAL_NEG_RATE = 0.02               # val: binary_classification 과 같은 표본
SEED = 42
assert P.edge.exists() and P.nodes.exists(), "04, 05 먼저"
con = D.connect(P, THREADS, "6GB")
D.register_labels(con, P)
PLAN = {"train": dict(neg_rate=TRAIN_NEG_RATE), "val": dict(keep=D.val_sample(con, P, VAL_NEG_RATE)), "test": dict()}

## 1. 구간별 저장

이미 있는 구간은 건너뛴다 (다시 만들려면 `lgbm/<구간>.parquet` 을 지우고 실행).
`rows_full` 이 binary_classification 실험의 구간 거래 수와 같아야 같은 모집단이다.

In [ ]:
summary = {}
for split, kw in PLAN.items():
    out = P.lgbm / f"{split}.parquet"
    if out.exists():
        print("이미 있음 → 건너뜀:", out.name); continue
    t0 = time.time()
    st = D.build_split(con, P, split, seed=SEED, verbose=False, **kw)
    st["seconds"] = round(time.time() - t0)
    summary[split] = st
    print(f"{split:8s} 전체 {st['rows_full']:>12,} (기대 {D.EXPECTED_ROWS[split]:,})  저장 {st['rows_kept']:>11,}  "
          f"양성 {st['positives']:,} (무패턴 {st['unmatched']:,})  {st['seconds']}s")
    assert st["rows_full"] == D.EXPECTED_ROWS[split], "binary_classification 실험과 구간 거래 수가 다르다"

## 2. 확인 — 파일 크기·열, 구간별 양성 구성, 무패턴 비교 피처

In [ ]:
import pyarrow.parquet as pq
rows = []
for split in PLAN:
    f = P.lgbm / f"{split}.parquet"; m = pq.ParquetFile(f).metadata
    lab = pd.read_parquet(f, columns=["y", "cls", "w"])
    rows.append(dict(split=split, rows=m.num_rows, cols=m.num_columns, GB=round(f.stat().st_size / 1e9, 2),
                     weighted_rows=round(lab.w.sum()), positives=int(lab.y.sum()), pattern=int(((lab.cls > 0) & (lab.cls < 9)).sum()),
                     nonpattern=int((lab.cls == 9).sum())))
pd.DataFrame(rows).set_index("split")

In [ ]:
# 패턴 유형별 건수 (cls 1~8, 9 = 무패턴)
lab = pd.concat({s: pd.read_parquet(P.lgbm / f"{s}.parquet", columns=["cls"]).cls.value_counts() for s in PLAN}, axis=1).fillna(0).astype(int)
lab.index = [D.PATTERN_NAMES[i] for i in lab.index]
lab.sort_index()

In [ ]:
# 비교 분석(nonpaten_vs_normal)에서 차이 난 피처를 test 구간에서 — 정상 / 패턴 / 무패턴 중앙값
cols = ["pair_prior_count", "receiver_gap_minutes", "paid_currency_z", "dst_log_in_count", "dst_log_active_days",
        "src_log_out_neighbors", "same_entity"]
t = pd.read_parquet(P.lgbm / "test.parquet", columns=["cls"] + cols)
t["group"] = np.select([t.cls == 0, t.cls == 9], ["normal", "nonpattern"], "pattern")
t.groupby("group")[cols].median().T.round(3)

In [ ]:
if summary:
    D.write_meta(P, "06_lgbm", dict(summary=summary, train_neg_rate=TRAIN_NEG_RATE, val_neg_rate=VAL_NEG_RATE, seed=SEED, created=time.strftime("%Y-%m-%d %H:%M:%S")))
con.close()
print("다음: 07 LightGBM (D.GNN_PARITY_COLUMNS 로 GNN 과 같은 조건, D.MODEL_COLUMNS 로 제외 3개 포함)")